# SENTINEL on MongoDB — Base Building Track

One notebook, five sections:

1. **Schema design** — `customers` (devices embedded) and `transactions` (customer referenced)
2. **Load data** — 5,000 SENTINEL transactions with customer, merchant and device fields
3. **CRUD** — `insert_one`, `insert_many`, `find` with filter + projection, `update_many`, `delete_one`
4. **Indexes** — on `customer_id` and `transaction_time`, with `explain()` before and after
5. **Aggregations** — fraud rate per merchant category; top 10 customers by amount in their last 24 h

Section 1 explains why devices are embedded and customers are referenced.

## Imports

In [ ]:
# pip install "pymongo[srv]" pandas supabase python-dotenv
import os
from datetime import datetime, timedelta, timezone
from decimal import Decimal
from dotenv import load_dotenv
from supabase import create_client
from pymongo import MongoClient, ReplaceOne, ASCENDING, DESCENDING
from pymongo.errors import BulkWriteError
from bson.decimal128 import Decimal128
from pprint import pprint

import pandas as pd

## Load credentials

All secrets live in a `.env` file next to this notebook (never in the notebook itself):

```
SUPABASE_URL=https://<project>.supabase.co
SUPABASE_KEY=<publishable key>
MONGODB_URI=mongodb+srv://<user>:<password>@<cluster>/
MONGODB_DATABASE=sentinel_db
```

In [ ]:
load_dotenv()

SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_KEY")
MONGODB_URI = os.getenv("MONGODB_URI")
MONGODB_DATABASE = os.getenv("MONGODB_DATABASE")

if not all([SUPABASE_URL, SUPABASE_KEY, MONGODB_URI, MONGODB_DATABASE]):
    raise ValueError(
        "Missing credentials. Check SUPABASE_URL, SUPABASE_KEY, "
        "MONGODB_URI and MONGODB_DATABASE in your .env file."
    )

## Connect to Supabase and MongoDB Atlas

In [ ]:
supabase = create_client(SUPABASE_URL, SUPABASE_KEY)
print("Supabase connected successfully!")

mongo_client = MongoClient(MONGODB_URI, serverSelectionTimeoutMS=15000)
mongo_client.admin.command("ping")

db = mongo_client[MONGODB_DATABASE]
customers_col = db["customers"]
transactions_col = db["transactions"]

print("Connected to Supabase and MongoDB Atlas.")
print("MongoDB database:", MONGODB_DATABASE)

## 1. Schema design

### `customers` — devices **embedded**
```json
{
  "_id": "C_ef13d0be3f91f70c",
  "customer_id": "C_ef13d0be3f91f70c",
  "created_at": ISODate("2020-06-27T18:30:00Z"),
  "customer_age": 60,
  "home_city": "Jaipur",
  "region": "north",
  "is_synthetic": true,
  "devices": [
    {"device_id": "D_c90626db699cc2ed", "device_kind": "browser", "is_synthetic": true,
     "first_seen": ISODate(...), "last_seen": ISODate(...), "transaction_count": 4}
  ]
}
```

### `transactions` — customer **referenced**, merchant and device **embedded**
```json
{
  "_id": "TX_0000000001",
  "transaction_id": "TX_0000000001",
  "customer_id": "C_...",                 // reference -> customers._id
  "merchant_id": "M_...",
  "device_id": "D_...",
  "merchant": {
    "merchant_id": "M_...", "synthetic_name": "Synthetic Merchant 000001",
    "merchant_category": "food",
    "merchant_location": {"city": "Mumbai", "lat": 19.08, "lon": 72.88},
    "is_synthetic": true
  },
  "device": {"device_id": "D_...", "device_kind": "mobile", "is_synthetic": true},
  "amount_paise": 327303,
  "transaction_amount": NumberDecimal("3273.03"),
  "payment_channel": "UPI",
  "merchant_category": "food",
  "transaction_time": ISODate("2024-10-01T..."),
  "customer_age": 62,
  "customer_location": {"city": "...", "lat": ..., "lon": ...},
  "merchant_location": {"city": "...", "lat": ..., "lon": ...},
  "account_age": 1154, "previous_transactions": 2,
  "average_transaction_value": NumberDecimal("1649.43"), "failed_transactions": 0,
  "split": "warmup", "is_synthetic": true,
  "fraud_label": 0
}
```

**Why:**
- **Devices are embedded** in the customer: a customer has only a handful of devices, they are always read with the customer
  ("which devices does this person use?"), and they are not shared or updated on their own.
- **Transactions reference the customer** by `customer_id`: transactions grow without limit (embedding them would hit the
  16 MB document limit and make every customer read heavy), and most queries scan transactions by time, merchant or label,
  not by customer.
- **Merchant and device are embedded in each transaction**: the fraud aggregations need the merchant category
  without a `$lookup`, and the copy records what was true at the moment of payment.

## 2. Load SENTINEL data from Supabase

Source: **your own Supabase project** from Project Work 4/5 (loaded with `sql/01_create_tables.sql` +
`python ingest.py all`). We read 5,000 transactions in ID order, then fetch only the related rows from
`customers`, `merchants`, `devices` and `labels`.

### Helper functions

In [ ]:
def fetch_transaction_sample(limit=5000, page_size=1000):
    """
    Fetch up to limit transactions in deterministic ID order
    """
    rows = []
    start = 0

    while len(rows) < limit:
        current_size = min(page_size, limit - len(rows))
        response = (
            supabase.table("transactions")
            .select("*")
            .order("transaction_id")
            .range(start, start + current_size - 1)
            .execute()
        )
        batch = response.data or []
        rows.extend(batch)

        if len(batch) < current_size:
            break

        start += len(batch)

    if len(rows) < limit:
        raise ValueError(
            f"Only {len(rows)} transactions were returned; "
            f"requires at least {limit}."
        )

    return rows[:limit]

In [ ]:
TARGET_TRANSACTIONS = 5000
transactions_raw = fetch_transaction_sample(TARGET_TRANSACTIONS)

print("Transactions fetched:", len(transactions_raw))

print("Example transaction:")
pprint(transactions_raw[0])

In [ ]:
def fetch_by_ids(table_name, id_field, ids, batch_size=200):
    """
    Fetch matching records in batches to avoid oversized IN filters
    """
    ids = list(set(ids))
    result = []

    for start in range(0, len(ids), batch_size):
        id_batch = ids[start:start + batch_size]
        if not id_batch:
            continue

        response = (
            supabase.table(table_name)
            .select("*")
            .in_(id_field, id_batch)
            .execute()
        )
        result.extend(response.data or [])

    return result

In [ ]:
customer_ids = {row["customer_id"] for row in transactions_raw}
merchant_ids = {row["merchant_id"] for row in transactions_raw}
device_ids = {
    row["device_id"]
    for row in transactions_raw
    if row.get("device_id") is not None
}
transaction_ids = {row["transaction_id"] for row in transactions_raw}

customers_raw = fetch_by_ids("customers", "customer_id", customer_ids)
merchants_raw = fetch_by_ids("merchants", "merchant_id", merchant_ids)
devices_raw = fetch_by_ids("devices", "device_id", device_ids)
labels_raw = fetch_by_ids("labels", "transaction_id", transaction_ids)

print("Related customers:", len(customers_raw))
print("Related merchants:", len(merchants_raw))
print("Related devices:", len(devices_raw))
print("Related label rows:", len(labels_raw))

### Conversion helpers

In [ ]:
def to_datetime(value):
    """
    Convert Supabase ISO timestamp strings to timezone-aware datetimes
    """
    if value is None or isinstance(value, datetime):
        return value

    try:
        parsed = datetime.fromisoformat(value.replace("Z", "+00:00"))
    except ValueError:
        # Python < 3.11 rejects fractions like ".6" -> let pandas parse it
        parsed = pd.Timestamp(value).to_pydatetime()
    if parsed.tzinfo is None:
        parsed = parsed.replace(tzinfo=timezone.utc)
    return parsed


def to_decimal128(value):
    """
    Preserve decimal currency values using BSON Decimal128.
    """
    if value is None:
        return None
    return Decimal128(Decimal(str(value)))


def to_binary_label(value):
    """
    Return a numeric 0/1 label, while rejecting unexpected values."""
    if value is None:
        return None
    if isinstance(value, bool):
        return int(value)
    label = int(value)
    if label not in (0, 1):
        raise ValueError(f"Expected fraud_label 0 or 1, got {value!r}")
    return label

### Build lookup maps and the embedded customer devices

In [ ]:
customer_map = {row["customer_id"]: row for row in customers_raw}
merchant_map = {row["merchant_id"]: row for row in merchants_raw}
device_map = {row["device_id"]: row for row in devices_raw}

label_map = {}
duplicate_label_transaction_ids = set()

for row in labels_raw:
    tx_id = row["transaction_id"]
    if tx_id in label_map:
        duplicate_label_transaction_ids.add(tx_id)
    label_map[tx_id] = row.get("fraud_label")

if duplicate_label_transaction_ids:
    raise ValueError(
        "Multiple label rows found for some transactions. "
        "Inspect labels_raw and define which label is authoritative. "
        f"Example IDs: {list(duplicate_label_transaction_ids)[:5]}"
    )

customer_devices = {}

for tx in transactions_raw:
    customer_id = tx["customer_id"]
    device_id = tx.get("device_id")

    if device_id and device_id in device_map:
        device = device_map[device_id]
        customer_devices.setdefault(customer_id, {})
        tx_time = to_datetime(tx["transaction_time"])

        if device_id not in customer_devices[customer_id]:
            customer_devices[customer_id][device_id] = {
                "device_id": device_id,
                "device_kind": device["device_kind"],
                "is_synthetic": device["is_synthetic"],
                "first_seen": tx_time,
                "last_seen": tx_time,
                "transaction_count": 0,
            }

        entry = customer_devices[customer_id][device_id]
        entry["first_seen"] = min(entry["first_seen"], tx_time)
        entry["last_seen"] = max(entry["last_seen"], tx_time)
        entry["transaction_count"] += 1

print("Customers with at least one device:", len(customer_devices))

### Build transaction documents

In [ ]:
transaction_documents = []
missing_merchant_ids = []
missing_fraud_label_ids = []

for source_tx in transactions_raw:
    document = dict(source_tx)
    tx_id = document["transaction_id"]
    merchant_id = document["merchant_id"]

    merchant = merchant_map.get(merchant_id)
    if merchant is None:
        missing_merchant_ids.append(merchant_id)
        continue

    # Embed all merchant fields from Supabase's merchants table.
    document["merchant"] = {
        "merchant_id": merchant["merchant_id"],
        "synthetic_name": merchant["synthetic_name"],
        "merchant_category": merchant["merchant_category"],
        "merchant_location": merchant["merchant_location"],
        "is_synthetic": merchant["is_synthetic"],
    }

    # Embed the device used for this payment (None when the transaction has no device).
    device = device_map.get(document.get("device_id"))
    document["device"] = (
        {
            "device_id": device["device_id"],
            "device_kind": device["device_kind"],
            "is_synthetic": device["is_synthetic"],
        }
        if device else None
    )

    # Attach fraud label using transaction_id.
    if tx_id in label_map and label_map[tx_id] is not None:
        document["fraud_label"] = to_binary_label(label_map[tx_id])
    else:
        document["fraud_label"] = None
        missing_fraud_label_ids.append(tx_id)

    document["transaction_time"] = to_datetime(
        document.get("transaction_time")
    )
    document["transaction_amount"] = to_decimal128(document.get("transaction_amount"))
    document["average_transaction_value"] = to_decimal128(
        document.get("average_transaction_value")
    )
    document["_id"] = tx_id

    transaction_documents.append(document)

print("Transaction documents prepared:", len(transaction_documents))
print("Missing merchants:", len(missing_merchant_ids))
print("Missing fraud labels:", len(missing_fraud_label_ids))
pprint(transaction_documents[0])

### Build customer documents

In [ ]:
customer_documents = []

for customer_id, source_customer in customer_map.items():
    document = dict(source_customer)
    document["_id"] = customer_id

    if document.get("created_at"):
        document["created_at"] = to_datetime(document["created_at"])

    document["devices"] = list(
        customer_devices.get(customer_id, {}).values()
    )

    customer_documents.append(document)

print("Customer documents prepared:", len(customer_documents))
pprint(customer_documents[0] if customer_documents else "No customers found")

### Load the real data into the two target collections

In [ ]:
customer_ops = [
    ReplaceOne({"_id": doc["_id"]}, doc, upsert=True)
    for doc in customer_documents
]

transaction_ops = [
    ReplaceOne({"_id": doc["_id"]}, doc, upsert=True)
    for doc in transaction_documents
]

try:
    customer_result = customers_col.bulk_write(customer_ops, ordered=False)
    transaction_result = transactions_col.bulk_write(transaction_ops, ordered=False)
except BulkWriteError as e:
    print("Bulk write error:", e.details["writeErrors"][:3])
    raise

print("Customers   -> inserted:", customer_result.upserted_count,
      "| updated:", customer_result.modified_count)
print("Transactions -> inserted:", transaction_result.upserted_count,
      "| updated:", transaction_result.modified_count)

print("customers in MongoDB   :", customers_col.count_documents({}))
print("transactions in MongoDB:", transactions_col.count_documents({}))
print("fraud transactions     :", transactions_col.count_documents({"fraud_label": 1}))

## 3. CRUD operations

### insert_one — add one new transaction for an existing customer

In [ ]:
sample_customer = customers_col.find_one({"devices.0": {"$exists": True}}, {"_id": 1, "devices": 1})
cid = sample_customer["_id"]
dev = sample_customer["devices"][0]

demo_tx = {
    "_id": "TX_DEMO_0001",
    "transaction_id": "TX_DEMO_0001",
    "customer_id": cid,
    "merchant_id": "M_DEMO",
    "device_id": dev["device_id"],
    "merchant": {"merchant_id": "M_DEMO", "synthetic_name": "Demo Merchant",
                 "merchant_category": "electronics",
                 "merchant_location": {"city": "Pune", "lat": 18.52, "lon": 73.86},
                 "is_synthetic": True},
    "device": {"device_id": dev["device_id"], "device_kind": dev["device_kind"],
               "is_synthetic": dev["is_synthetic"]},
    "merchant_category": "electronics",
    "amount_paise": 2500000,
    "transaction_amount": Decimal128("25000.00"),
    "payment_channel": "card",
    "transaction_time": datetime.now(timezone.utc),
    "fraud_label": 0,
    "label_source": "demo",
    "split": "demo",
}
result = transactions_col.insert_one(demo_tx)
print("inserted_id:", result.inserted_id)

### insert_many — add several demo transactions at once

In [ ]:
more_demo = [
    {**demo_tx, "_id": f"TX_DEMO_000{i}", "transaction_id": f"TX_DEMO_000{i}",
     "amount_paise": amt * 100, "transaction_amount": Decimal128(f"{amt}.00")}
    for i, amt in [(2, 1200), (3, 45000), (4, 799)]
]
result = transactions_col.insert_many(more_demo)
print("inserted_ids:", result.inserted_ids)

### find — filter + projection

In [ ]:
# Electronics transactions above Rs 20,000 paid by UPI, showing only a few fields
query = {
    "merchant.merchant_category": "electronics",   # field inside the embedded merchant
    "transaction_amount": {"$gt": 20000},
    "payment_channel": "UPI",
}
projection = {"_id": 0, "transaction_id": 1, "customer_id": 1,
              "transaction_amount": 1, "transaction_time": 1, "fraud_label": 1}

print("matches:", transactions_col.count_documents(query))
for doc in transactions_col.find(query, projection).sort("transaction_amount", DESCENDING).limit(5):
    print(doc)

In [ ]:
# A customer with their embedded devices (projection on an array)
pprint(customers_col.find_one({"_id": cid}, {"home_city": 1, "region": 1, "devices.device_id": 1, "devices.device_kind": 1}))

### update_many — flag high-value transactions for review

In [ ]:
result = transactions_col.update_many(
    {"transaction_amount": {"$gte": 50000}},
    {"$set": {"review_flag": "high_value"}},
)
print("matched:", result.matched_count, "| modified:", result.modified_count)
print("example:", transactions_col.find_one({"review_flag": "high_value"},
                                             {"_id": 0, "transaction_id": 1, "transaction_amount": 1, "review_flag": 1}))

### delete_one — remove the demo transaction

In [ ]:
result = transactions_col.delete_one({"_id": "TX_DEMO_0001"})
print("deleted:", result.deleted_count)
print("still there?", transactions_col.find_one({"_id": "TX_DEMO_0001"}))

# Tidy up the other demo rows so they don't affect sections 4 and 5
transactions_col.delete_many({"split": "demo"})
print("transactions:", transactions_col.count_documents({}))

## 4. Indexes and explain()

Query under test: *all transactions of one customer in a time window, newest first* — the typical
"show me this customer's recent activity" lookup a fraud analyst runs.

In [ ]:
def explain_summary(cursor):
    ex = cursor.explain()
    stats = ex["executionStats"]
    stages, plan = [], ex["queryPlanner"]["winningPlan"]
    while plan:
        stages.append(plan["stage"] + (f" ({plan['indexName']})" if "indexName" in plan else ""))
        plan = plan.get("inputStage") or (plan.get("inputStages") or [None])[0]
    print("plan            :", " <- ".join(stages))
    print("nReturned       :", stats["nReturned"])
    print("docs examined   :", stats["totalDocsExamined"])
    print("keys examined   :", stats["totalKeysExamined"])
    print("time (ms)       :", stats["executionTimeMillis"])

busy_customer = next(transactions_col.aggregate([
    {"$group": {"_id": "$customer_id", "n": {"$sum": 1}}},
    {"$sort": {"n": -1}}, {"$limit": 1}]))["_id"]

# time window: the 30 days up to this customer's latest transaction
window_end = transactions_col.find_one({"customer_id": busy_customer},
                                       sort=[("transaction_time", DESCENDING)])["transaction_time"]
window_start = window_end - timedelta(days=30)

def customer_window_query():
    return (transactions_col
            .find({"customer_id": busy_customer,
                   "transaction_time": {"$gte": window_start, "$lte": window_end}},
                  {"_id": 0, "transaction_id": 1, "transaction_amount": 1, "transaction_time": 1})
            .sort("transaction_time", DESCENDING))

print("customer:", busy_customer, "| window:", window_start, "->", window_end)

### Before indexes

In [ ]:
transactions_col.drop_indexes()          # only the default _id index remains
print(list(transactions_col.index_information()))
explain_summary(customer_window_query())

Expect `COLLSCAN`: MongoDB reads every one of the ~5,000 documents to find this customer's few rows.

### Create the indexes

In [ ]:
transactions_col.create_index([("customer_id", ASCENDING)], name="idx_customer_id")
transactions_col.create_index([("transaction_time", DESCENDING)], name="idx_transaction_time")
pprint(transactions_col.index_information())

### After indexes

In [ ]:
explain_summary(customer_window_query())

Now the plan is `IXSCAN (idx_customer_id)` → `FETCH`: docs examined drops from ~5,000 to just this customer's rows.

**Optional, better for this exact query:** a compound index `{customer_id: 1, transaction_time: -1}` covers the
equality, the range and the sort in one index, so no in-memory sort is needed.

In [ ]:
transactions_col.create_index([("customer_id", ASCENDING), ("transaction_time", DESCENDING)],
                              name="idx_customer_time")
explain_summary(customer_window_query())

## 5. Aggregation pipelines

### 5a. Fraud rate per merchant category

In [ ]:
fraud_rate_pipeline = [
    {"$match": {"fraud_label": {"$ne": None}}},   # only labelled transactions
    {"$group": {
        "_id": "$merchant.merchant_category",
        "transactions": {"$sum": 1},
        "fraud_count": {"$sum": "$fraud_label"},
        "total_amount": {"$sum": "$transaction_amount"},
    }},
    {"$project": {
        "_id": 0,
        "merchant_category": "$_id",
        "transactions": 1,
        "fraud_count": 1,
        "fraud_rate_pct": {"$round": [{"$multiply": [{"$divide": ["$fraud_count", "$transactions"]}, 100]}, 3]},
        "total_amount": {"$toDouble": {"$round": ["$total_amount", 2]}},
    }},
    {"$sort": {"fraud_rate_pct": -1}},
]
fraud_by_category = pd.DataFrame(list(transactions_col.aggregate(fraud_rate_pipeline)))
fraud_by_category

### 5b. Top 10 customers by amount in their last 24 hours of activity

"Last 24 hours" is per customer: find each customer's latest transaction, then add up everything they spent in
the 24 hours up to and including it.

In [ ]:
top_24h_pipeline = [
    {"$group": {
        "_id": "$customer_id",
        "last_txn_time": {"$max": "$transaction_time"},
        "txns": {"$push": {"t": "$transaction_time", "amt": "$transaction_amount"}},
    }},
    {"$project": {
        "last_txn_time": 1,
        "window": {"$filter": {
            "input": "$txns", "as": "x",
            "cond": {"$gt": ["$$x.t", {"$subtract": ["$last_txn_time", 24 * 60 * 60 * 1000]}]},
        }},
    }},
    {"$project": {
        "_id": 0,
        "customer_id": "$_id",
        "last_txn_time": 1,
        "txns_in_last_24h": {"$size": "$window"},
        "amount_last_24h": {"$toDouble": {"$round": [{"$sum": "$window.amt"}, 2]}},
    }},
    {"$sort": {"amount_last_24h": -1}},
    {"$limit": 10},
    # bring in a few customer fields via the reference
    {"$lookup": {"from": "customers", "localField": "customer_id", "foreignField": "_id", "as": "c"}},
    {"$set": {"home_city": {"$first": "$c.home_city"},
              "devices_used": {"$size": {"$ifNull": [{"$first": "$c.devices"}, []]}}}},
    {"$unset": "c"},
]
top_customers_24h = pd.DataFrame(list(transactions_col.aggregate(top_24h_pipeline)))
top_customers_24h

## Summary

| Section | What it shows |
|---|---|
| 1 | `customers` embeds devices; `transactions` references the customer and embeds merchant/device snapshots |
| 2 | 5,000 SENTINEL transactions + related customers, merchants, devices and labels read from Supabase, loaded with `ReplaceOne` upserts |
| 3 | `insert_one`, `insert_many`, `find` (filter + projection), `update_many`, `delete_one` |
| 4 | `COLLSCAN` → `IXSCAN` after indexing `customer_id` and `transaction_time` |
| 5 | Fraud rate per `merchant.merchant_category`; top 10 customers by spend in their last 24 h |

In [ ]:
mongo_client.close()